In [ ]:
from astropy.io import fits
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 


obs_id = "m3g20090118t022705"

In [ ]:
metadata = pd.read_csv("/home/bekah/m3-pipeline-dev/obs_to_cal_file_mapping/obs_cal_info.csv") 
metadata = metadata[metadata['obs_id'] == obs_id.upper()]

if len(metadata) == 0:
    print("This is not a valid observation ID, although it could be a dark"
          " signal observation.")
elif len(metadata) > 1:
    best_idx = metadata['version'].str.extract(r'(\d+)')[0].astype(int).idxmax()
    metadata = metadata.loc[[best_idx]] 

dark_id = metadata['dark_signal_id'].iloc[0].lower()
flat_id = metadata['flat_field_id'].iloc[0].lower()
bde_id = metadata['bad_detector_map_id'].iloc[0].lower()

print(f"Temp: {metadata['obs_temperature']}")
print(f"Dark ID: {dark_id}")
print(f"Flat ID: {flat_id}")
print(f"BDE ID: {bde_id}")

# load dark signal observation 
dark_path = f"/home/bekah/m3-pipeline-dev/data/dark/darks_global/{dark_id}_l0.fits"
with fits.open(dark_path) as hdul:
    dark = hdul[0].data.transpose(1,0,2)[5:-5, :, :]# bad stuff at ends  
dark = np.mean(dark, axis=0) 

# load observation 
obs_path = f"/home/bekah/m3-pipeline-dev/matching_area_obs/{obs_id}_l0.fits"
with fits.open(obs_path) as hdul:
    obs_image = hdul[0].data.transpose(1,0,2)[5:-5, :, :]

# subtract dark signal 
obs_image = obs_image - dark[np.newaxis, :, :]

obs_image = obs_image.transpose(1,0,2)[:, :, :] 

nbands, nlines, nsamples = obs_image.shape
print(f"Cube shape: {obs_image.shape}")
        

In [ ]:
del dark
del metadata

In [ ]:

cube = obs_image
neg_mask = obs_image < 0

neg_counts = neg_mask.sum(axis=2)  

n_bands, n_lines, n_samples = cube.shape
neg_counts = np.empty((n_bands, n_lines), dtype=np.int32)
neg_means = np.full((n_bands, n_lines), np.nan, dtype=np.float32)

for b in range(n_bands):
    band = cube[b]                   
    mask_b = band > 0                
    counts_b = mask_b.sum(axis=1)      
    sums_b = np.where(mask_b, band, 0).sum(axis=1)  

    neg_counts[b] = counts_b
    valid = counts_b > 0
    neg_means[b, valid] = sums_b[valid] / counts_b[valid]


In [ ]:

n_bands, n_lines, n_samples = cube.shape
bands_arr = np.arange(n_bands)

avg_1_3_per_line = cube[:, :, 1:3].mean(axis=(0, 2))
avg_318_320_per_line = cube[:, :, 318:320].mean(axis=(0, 2))

neg_counts_orig = np.empty((n_bands, n_lines), dtype=np.int32)
for b in range(n_bands):
    neg_counts_orig[b] = (cube[b, :, 20:300] < 0).sum()#(axis=0,1)
total_orig = neg_counts_orig.sum(axis=1)

def baseline_0_3(b):
    return cube[b, :, 0:3].mean(axis=1)

def baseline_1_3(b):
    return cube[b, :, 1:3].mean(axis=1)

def baseline_1(b):
    #return cube[b, :, 1]
    med = np.median(cube[b, :, 20:300], axis=1) * .07
    return -np.where(med > 0, med, 0)

def baseline_318_320(b):
    return cube[b, :, 318:320].mean(axis=1) 

def baseline_0_3_and_318_320(b):
    edge = np.concatenate([cube[b, :, 0:3], cube[b, :, 318:320]], axis=1)
    return edge.mean(axis=1) 

def baseline_1_3_and_318_320(b):
    return (avg_1_3_per_line + avg_318_320_per_line) / 2

configs = [
     # ("[0:3]", baseline_0_3),
     # ("[1:3]", baseline_1_3),
     # ("[318:320]", baseline_318_320),
     ("[0:3] and [318:320]", baseline_0_3_and_318_320),
    ("[1:3] and [318:320] (per line)", baseline_1_3_and_318_320),
     ("% of med illum", baseline_1), 
 ]

totals_corrected = {}
for name, baseline_fn in configs:
    neg_counts_corr = np.empty((n_bands), dtype=np.float32)
    for b in range(n_bands):
        band = cube[b]
        baseline = baseline_fn(b)
        corrected = band - baseline[:, None] 
        corrected = corrected[:, 20:300]
        #corrected = corrected + corrected[:, 4:7].mean(axis=1)[:, None]
        neg_counts_corr[b] = np.median(corrected[corrected < 0])
    totals_corrected[name] = neg_counts_corr

okabe_ito = [
     "#E69F00",  # orange
     "#56B4E9",  # sky blue
     "#009E73",  # bluish green
     "#F0E442",  # yellow
    "#0072B2",  # blue
     "#D55E00",  # vermillion
     "#CC79A7",  # reddish purple
]

linestyles = [(0, (5, 1)), "-", "--", "-.", ":", (0, (3, 1, 1, 1))] 

In [ ]:
plt.figure(figsize=(12, 7))


#plt.plot(bands_arr, total_orig, label="dss counts", color="black", linestyle=(0, (1, 1)), lw=2, alpha=0.8)

for (name, _), color, ls in zip(configs, okabe_ito, linestyles):
    plt.plot(bands_arr, totals_corrected[name], label=f"shift mean{name}",
              color=color, linestyle=ls, lw=2)

plt.ylim(0, -1)
plt.xlabel("band")
plt.ylabel("med negative pixels (all lines)")
plt.title(f"dark pedestal correction options, {obs_id}")
plt.legend(fontsize=9, loc="upper right")
plt.savefig(f"{obs_id}_darkped_corr_opts_medneg.png")
plt.tight_layout()
plt.show()

In [ ]:
 np.median(corrected[corrected < 0])